# Dados e Aprendizagem Automática
### Part X

## **SVM using GenAudio Dataset**

For this class, we will use a music dataset. The [GenAudio dataset](https://reposlink.di.uminho.pt/uploads/55a7037e4e5dc4624a24a54398fccb5c.file.Data.zip) is composed by audio files and CSV files with extracted features from the audio files:
- **genres original** - collection of 10 genres with 100 audio files each, all having a length of 30 seconds
- **CSV files** - extracted features of the audio files. One file has for each song (30 seconds long) a mean and variance computed over multiple features that can be extracted from an audio file. The other file has the same structure, but the songs were split before into 3 seconds audio files (this way increasing 10 times the amount of data we fuel into our classification models).

This dataset is frequently used for evaluation in machine listening research for Music Genre Recognition (MGR). The files were collected in 2000-2001 from a variety of sources including personal CDs, radio, microphone recordings, in order to represent a variety of recording.

### Imports, installations and settings

In order to work with audio data, we will use [librosa](https://librosa.org/doc/main/index.html), a python library used for audio and music analysis. It is a powerful package widely used for audio visualization and for building Music Information Retrieval (MIR) systems.

Install *librosa*: <code>pip install librosa</code>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display as lplt
import IPython
import IPython.display as ipd

import sklearn
import os

from IPython.display import Audio
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline

### Explore the sound files

Select one file to start

In [ ]:
audio_path = 'Data/genres_original/'

*Note*: Understanding waveform, sampling rate and other sound concepts: [here](https://medium.com/analytics-vidhya/understanding-the-mel-spectrogram-fca2afa2ce53)

Load the audio as a waveform <code>data</code> and store the sampling rate as <code>sr</code>

In [ ]:
data, sr = librosa.
print(type(data), type(sr))

print(data., sr)

Initializing the sample rate to 45600 so we can obtain the signal value array

In [ ]:
librosa.load(audio_path, )

Taking Short-Time Fourier Transform (SFT) of the signal

In [ ]:
stft = librosa.(data)

Convert an amplitude spectrogram to dB-scaled spectrogram

In [ ]:
stft_db = librosa.(abs(stft))

Playing the audio file

In [ ]:
IPython.(data, rate=sr)

It is important to note that while working with any kind of audio data to solve any kind of problem statement, using only <code>.wav</code> format audio files is appropriate to analyze the data. If you are given audio files with <code>.mp3</code> format you have to batch convert the data to waveforms using online software as <code>.wav</code> is the standard way of representing the audio files and it is the only way to work with audio data.

Let's see the wave form representation of this audio file

In [ ]:
plt.figure(figsize=(7, 4))
librosa.display.waveshow(, color="#2B4F72", alpha=0.5)
plt.()

A spectrogram is a visual representation of the signal loudness over time at different frequencies included in a certain waveform. We can examine increase or decrease of energy over period of time. Spectrograms are also known as sonographs, voiceprints, and voicegrams.

Let's convert the song into image using *Spectrogram*

In [ ]:
plt.figure(figsize=(7, 6))
librosa.display.(stft_db, sr=sr, x_axis='time', y_axis='hz')
plt.()

### Data Pre-Processing

##### Extracting Audio Features

The process of extraction of features from the data to utilize them for analysis is
known as feature extraction. Each audio signal consists of various audio features
however we must extract features that are relevant to the problem that we are solving.
Here are some features listed which are used in our project.

**Spectral Roll-Off** 
It computes the roll-off frequency for each frame in each signal. The frequency under which some percentage (*cut-off*) of the total energy of a spectrum is obtained. It can be used to differentiate between the harmonic and noisy sounds.

In [ ]:
spectral_rolloff = librosa.(y=data, sr=sr)[0]

plt.figure(figsize=(7, 6))
librosa.display.waveshow(, sr=sr, alpha=0.4, color="#2B4F72")

**Chroma Feature**
It closely relates with the twelve different pitch classes. Chroma based features are also called as pitch class profiles. It is the powerful tool for analyzing and categorizing them. Harmonic and melodic characteristics of music are captured by them. It computes the chromogram from a waveform or power spectrogram.

In [ ]:
chroma = librosa.(y=data, sr=sr)

plt.figure(figsize=(7, 4))
lplt.specshow(chroma, sr=sr, x_axis="time", y_axis="chroma", cmap="BuPu")
plt.colorbar()
plt.title("Chroma Features")
plt.()

**Zero-Crossing Rate**
The Zero-Crossing Rate (ZCR) is the rate of sign-changes along with a signal, i.e., the rate at which the signal changes from positive to negative or back - the number of times the signal crosses x-axis. This feature is heavily used in both speech recognition and music information retrieval. It usually has higher values for highly percussive sounds like those in metal and rock.

In [ ]:
n0 = 
n1 = 
plt.figure(figsize=(14, 5))
plt.plot(data[], color="#2B4F72")
plt.grid()

zcr = librosa.(data[n0:n1], pad=False)
print(sum(zcr))

Changing the start and the end

In [ ]:
start = 
end = 
plt.figure(figsize=(12, 4))
plt.plot(data[], color="#2B4F72")

zcr = librosa.(data[start:end], pad=False)
print(sum(zcr))

**Spectral Centroid**
It indicates where the "center of mass" for a sound is located and is calculated as the weighted mean of the frequencies present in the sound. Consider two songs, one from a blues genre and the other belonging to metal. Now, as compared to the blues genre song, which is the same throughout its length, the metal song has more frequencies towards the end. So spectral centroid for blues song will lie somewhere near the middle of its spectrum while that for a metal song would be towards its end. To compute the spectral centroid, each frame of a magnitude spectrogram is normalized and treated as a distribution over frequency bins, from which the mean (centroid) is extracted per frame.

In [ ]:
spectral_centroids = librosa.(y=data, sr=sr)[0]
spectral_centroids.

Convert frame counts to time (seconds)

In [ ]:
 = range(len(spectral_centroids))
t = librosa.frames)

Transform features by scaling each feature to a given range:

<code>sklearn.preprocessing.minmax_scale</code> scales and translates each feature individually such that it is in the given range on the training set, i.e. between zero and one.
This transformation is often used as an alternative to zero mean, unit variance scaling.

In [ ]:
def normalize(data, axis=0):
    return sklearn.(data, axis=axis)

In [ ]:
librosa.display.waveshow(data, sr=sr, alpha=0.4)
plt.(t, normalize(spectral_centroids), color='r')

### Exploratory Data Analysis (EDA)
Vizualizing the audio files, wave plots and spectrograms for all the 10 genre classes

**1. BLUES**

In [ ]:
 = 'Data/genres_original/blues/blues.00001.wav'
data, sr = librosa.()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr, alpha=0.4,)
plt.title('Waveplot - ')

Creating log mel spectrogram

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.(y=data, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.(spectrogram)
librosa.display.(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - BLUES')
plt.colorbar(format='%+2.0f dB')

Playing the audio

In [ ]:
ipd.Audio() 

**2. CLASSICAL**

In [ ]:
audio2 = 'Data/genres_original/classical/classical.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr, alpha=0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram -')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio() 

**3. COUNTRY**

In [ ]:
 = 'Data/genres_original/country/country.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr, alpha=0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - ')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio()

**4. DISCO**

In [ ]:
 = 'Data/genres_original/disco/disco.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr, alpha=0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - ')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio() 

**5. HIPHOP**

In [ ]:
 = 'Data/genres_original/hiphop/hiphop.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr, alpha = 0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000,) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - ')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio()

**6. JAZZ**

In [ ]:
 = 'Data/genres_original/jazz/jazz.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr, alpha=0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - ')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio()

**7. METAL**

In [ ]:
 = 'Data/genres_original/metal/metal.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr,alpha=0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - ')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio()

**8. POP**

In [ ]:
 = 'Data/genres_original/pop/pop.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(8, 3))
librosa.display.waveshow(data, sr=sr, alpha=0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - ')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio()

**9. REGGAE**

In [ ]:
 = 'Data/genres_original/reggae/reggae.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr, alpha=0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - ')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio()

**10. ROCK**

In [ ]:
 = 'Data/genres_original/rock/rock.00001.wav'
data, sr = librosa.load()
plt.figure(figsize=(7, 3))
librosa.display.waveshow(data, sr=sr, alpha=0.4)
plt.title('Waveplot - ')

In [ ]:
plt.figure(figsize=(7, 4))
spectrogram = librosa.feature.melspectrogram(y=, sr=sr, n_mels=128, fmax=8000) 
spectrogram = librosa.power_to_db(spectrogram)
librosa.display.specshow(spectrogram, y_axis='mel', fmax=8000, x_axis='time')
plt.title('Mel Spectrogram - ')
plt.colorbar(format='%+2.0f dB')

In [ ]:
ipd.Audio()

### Train and Test Data

To create the X and y of this dataset, we need to process the data, extract the features and assign the labels. Let's load each file from each genre to X and the correpsonding genre (label) to y.

In [ ]:
path = 

Extract the features

In [ ]:
def extract_features():
    try:
        audio, sr = librosa.

        mfccs = librosa.(y=, sr=sr, n_mfcc=13)
        spectral_centroid = librosa.(y=, sr=sr)
        chroma = librosa.(y=, sr=sr)

        features = []
        for feature in [a]:
            features.extend([
                np.mean(feature),
                np.std(feature),
                np.max(feature),
                np.min(feature)
            ])
            
        return 
    except Exception as e:
        print(f"Error extracting features from {file_path}: {str(e)}")
        return None

Process the data

In [ ]:
def process_data():
    X = []
    y = []

    for genre in os.listdir():
        genre_path = os.path.join(, )
        if os.path.isdir(genre_path):
            print(f"Processing {genre} files...")
            
            for file_name in os.listdir():
                if file_name.endswith('.wav'):
                    file_path = os.path.join(genre_path, file_name)
                    extracted_features = extract_features(file_path)
                    
                    if extracted_features:
                        X.append(extracted_features)
                        y.append(genre)
    
    return np.array(), np.array()

In [ ]:
X, y = process_data()

In [ ]:
print(X., y.)

Now we need to split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(, test_size = , random_state = 42)

In [ ]:
print(X_train., y_train.)

In [ ]:
print(X_test., y_test.)

### Support Vector Classifier

<code>class sklearn.svm.SVC(*, C=1.0, kernel='rbf', degree=3, gamma='scale', coef0=0.0, shrinking=True, probability=False, tol=0.001, cache_size=200, class_weight=None, verbose=False, max_iter=-1, decision_function_shape='ovr', break_ties=False, random_state=None)</code>

Let's use default values:

In [ ]:
classifier = 

In [ ]:
classifier.fit()

Obtain the scores:

In [ ]:
print("Training set score: {:.3f}".format(classifier.(, )))
print("Test set score: {:.3f}".format(classifier.(, )))

Obtain predictions:

In [ ]:
y_pred = classifier.

Evaluate the model:

In [ ]:
cm = confusion_matrix()

['blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5 , 'metal': 6, 'pop': 7, 'reggae': 8, 'rock’: 9]

In [ ]:
sns.set(rc = {'figure.figsize':(9, 4)})
sns.heatmap(, annot=True)
print(classification_report())

#### Apply the SVC to one file

In [ ]:
path_test = 'Data/genres_original/'

In [ ]:
audio_test, sr = librosa.load()

In [ ]:
ipd.Audio(, rate=sr)

In [ ]:
a_test=[]
mfccs = librosa.feature.mfcc(y=, sr=sr, n_mfcc=13)
spectral_centroid = librosa.feature.spectral_centroid(y=, sr=sr)
chroma = librosa.feature.chroma_stft(y=, sr=sr)
features = []
for feature in []:
    features.extend([
         np.mean(feature),
        np.std(feature),
        np.max(feature),
        np.min(feature)
        ])
a_test.append()

In [ ]:
print(classifier.

### How can the model be improved?